In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, norm

MOSC_PATH = "../data/mosc_data.csv"
USGS_PATH = "../data/usgs_data/usgs_m4_2005_2025.csv"

def load_moscow():
    df = pd.read_csv(MOSC_PATH, parse_dates=["datetime"])
    return df.set_index("datetime").sort_index()["value"]

def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()

cr = load_moscow()
eq = load_earthquakes(min_mag=4.0)
print(f"CR: {len(cr)} pomiarów, {cr.index.min()} .. {cr.index.max()}")
print(f"EQ (M>=4.0): {len(eq)} zdarzeń, {eq.index.min()} .. {eq.index.max()}")

In [ ]:
def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    if n_total == 0:
        return dict(N=N, N_valid=0, Np=0, Nm=0, PPDF=np.nan, PCDF=np.nan, sigma=np.nan)

    ppdf = binom.pmf(Np, n_total, 0.5)
    pcdf = binom.sf(Np - 1, n_total, 0.5)
    sigma = norm.isf(pcdf)

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma)


def full_d_scan(cr, eq, t0, P_days, m, dt_days, d_range):
    return {d: cosmoseismic_stat(cr, eq, t0, P_days, d, m, dt_days)["PCDF"] for d in d_range}


def circular_shift_eq(eq, rng):
    start, end = eq.index.min(), eq.index.max()
    span = end - start
    shift = pd.Timedelta(seconds=int(rng.uniform(0, span.total_seconds())))
    new_idx = (start + ((eq.index - start + shift) % span)).astype(eq.index.dtype)
    return pd.Series(eq.values, index=new_idx).sort_index()


t0 = pd.Timestamp("2013-11-14 07:00:00")
rows = []
for d in range(1, 31):
    r = cosmoseismic_stat(cr, eq, t0, P_days=1675, d_days=d, m=4.0, dt_days=15)
    r["d"] = d
    rows.append(r)
scan = pd.DataFrame(rows).set_index("d")

best_d = scan["PCDF"].idxmin()
best_pcdf = scan["PCDF"].min()
print(f"Obserwowane minimum: PCDF={best_pcdf:.3e} przy d={best_d} (odtworzone z 20260715.ipynb, tylko definicje)")

In [ ]:
# Tak dokładnie policzono ../results/mc_minima_n100000.npy (odkomentuj, żeby powtórzyć
# od zera - zajmuje ~87 min na 20 rdzeniach CPU; multiprocessing zamiast
# threading, bo GIL serializuje kod Pythona/pandas na jednym rdzeniu
# niezależnie od liczby wątków - patrz mc_parallel.py i 20260716.txt).
# Poniższa komórka jest zakomentowana celowo: wynik jest już zapisany
# w ../results/mc_minima_n100000.npy (wczytywany w kolejnej komórce), żeby każdy mógł
# odtworzyć/zweryfikować wyniki bez ponownego czekania 87 minut, ale sam
# kod generujący ten plik zostaje tu widoczny do wglądu/reprodukcji.

# import sys, time
# sys.path.insert(0, "..")
# from mc_parallel import run_mc_parallel
#
# n_sims = 100000
# save_path = f"../results/mc_minima_n{n_sims}.npy"
# t_start = time.time()
# mc_minima = run_mc_parallel(
#     cr, eq, t0, 1675, 4.0, 15, n_sims,
#     full_d_scan_fn=full_d_scan, circular_shift_fn=circular_shift_eq,
#     base_seed=42, save_path=save_path,
# )
# elapsed = time.time() - t_start
# print(f"n_sims={n_sims} w {elapsed:.1f} s ({elapsed / n_sims * 1000:.1f} ms/symulację)")
# print(f"wynik zapisany do {save_path}")
#
# p_global = (mc_minima <= best_pcdf).mean()
# p_global_floor = max(p_global, 1 / n_sims)

In [ ]:
# Wczytanie gotowego wyniku Monte Carlo zamiast ponownego liczenia
# (n_sims=1e5 trwa ~87 min, patrz mc_parallel.run_mc_parallel + 20260716.txt).
# Plik wygenerowany raz przez run_mc_parallel(..., save_path="../results/mc_minima_n100000.npy")
# w tej samej konfiguracji (t0, P_days=1675, d=1..30, dt=15, m=4.0, base_seed=42).
n_sims = 100000
mc_minima = np.load(f"../results/mc_minima_n{n_sims}.npy")
print(f"wczytano {len(mc_minima)} symulacji z ../results/mc_minima_n{n_sims}.npy")

p_global = (mc_minima <= best_pcdf).mean()
p_global_floor = max(p_global, 1 / n_sims)

In [ ]:
print(f"Monte Carlo (n_sims={n_sims}): p_global (empiryczne) = {p_global:.4f}")
print(f"  najbardziej ekstremalne minimum spod H0: {mc_minima.min():.3e}")
print(f"  mediana minimów spod H0: {np.median(mc_minima):.3e}")
print(f"  floor (bo 0/{n_sims}): p_global <= {p_global_floor:.4f} (~{norm.isf(p_global_floor):.2f} sigma)")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(np.log10(mc_minima), bins=30, color="steelblue", edgecolor="black", alpha=0.8,
         label=f"min(PCDF) pod H0 (n={n_sims} symulacji)")
plt.axvline(np.log10(best_pcdf), color="red", ls="--",
            label=f"obserwowane min(PCDF)={best_pcdf:.1e} (d={best_d})")
plt.xlabel("log10(min PCDF w skanie d=1..30)")
plt.ylabel("liczba symulacji")
plt.legend()
plt.grid(ls=":")
plt.tight_layout()
plt.show()